# Debug HierarchicalForecast

## Importation des modules

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Importation des modules
# Modules de base
import numpy as np
import pandas as pd
import sys

# Ajout du chemin
sys.path.append('..')

# Importation des utilitaires sklearn
from sklearn.utils import _safe_indexing
from sklearn.utils.metaestimators import _safe_split
from sklearn.model_selection import cross_val_predict

# Importation des modèles
# Sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
# XGBoost

# Importation des pipelines
# Sklearn
from sklearn.pipeline import Pipeline

#  Eléments du package à intégrer
# Crossval
from tsforecast.crossvals import (
    PanelOutOfSampleSplit
)

## Création des données

### Création des données sources

In [ ]:
def generate_hierarchical_data(
    entities=['X', 'Y', 'Z'],
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
    seed=42,
):
    """Generate hierarchical panel data where value_a = value_b + value_c and value_c = value_d + value_e.

    Generates two bottom-level series (value_b, value_c) per entity, with
    distinct trend, seasonality and noise patterns. The top-level series
    (value_a) is their exact sum, forming a cross-sectional aggregation
    constraint. Data is indexed by (entity, date).

    Args:
        entities: List of entity identifiers.
        start_date: Start date for the time series.
        periods: Number of time periods per entity.
        freq: Frequency of observations.
        heterogeneous_effects: Whether entities have different baseline levels
            and trend slopes for value_b and value_c.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (entity, date) and columns value_a, value_b, value_c, value_d, value_e.
    """
    # Initialisation du seed
    np.random.seed(seed)
    # Initialisation des dates
    dates = pd.date_range(start=start_date, periods=periods, freq=freq)
    t = np.arange(periods)

    # Liste des observations hiérarchiques
    hierarchical_records = []

    # Parcours des entités
    for entity in entities:
        # Effets fixes par entité (niveaux et pentes)
        if heterogeneous_effects:
            base_b = np.random.uniform(40, 70)
            base_d = np.random.uniform(20, 45)
            base_e = np.random.uniform(10, 25)
            slope_b = np.random.uniform(0.05, 0.25)
            slope_d = np.random.uniform(-0.10, 0.05)
            slope_e = np.random.uniform(-0.20, 0.15)
        else:
            base_b, base_d, base_e = 50, 30, 10
            slope_b, slope_d, slope_e = 0.15, -0.05, -0.15

        # Composante B : tendance + saisonnalité annuelle + bruit
        value_b = (
            base_b
            + slope_b * t
            + 5 * np.sin(2 * np.pi * t / 12)
            + np.random.normal(0, 1.5, periods)
        )

        # Composante D : tendance + saisonnalité semestrielle + bruit
        value_d = (
            base_d
            + slope_d * t
            + 3 * np.sin(2 * np.pi * t / 6)
            + np.random.normal(0, 1.0, periods)
        )

        # Composante E : tendance + saisonnalité semestrielle + bruit
        value_e = (
            base_e
            + slope_e * t
            + 3 * np.sin(2 * np.pi * t / 6)
            + np.random.normal(0, 1.0, periods)
        )

        # Agrégat : contrainte d'agrégation exacte
        value_c = value_d + value_e
        value_a = value_b + value_c

        # Accumulation des lignes pour le format wide
        for d, va, vb, vc, vd, ve in zip(dates, value_a, value_b, value_c, value_d, value_e):
            hierarchical_records.append({
                'entity': entity, 'date': d,
                'value_a': va, 'value_b': vb, 'value_c': vc, 'value_d': vd, 'value_e': ve,
            })

    # Format wide : MultiIndex (entity, date)
    df_hierarchical = pd.DataFrame(hierarchical_records).set_index(['entity', 'date'])

    return df_hierarchical


# Génération des données hiérarchiques
print('📊 Génération de données hiérarchiques ...')
df_hier = generate_hierarchical_data(
    entities=['FR', 'DE', 'IT', 'ES'],
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
)

n_entities = df_hier.index.get_level_values('entity').nunique()
constraint_ok = (
    df_hier
    .assign(check=lambda d: np.isclose(d['value_a'], d['value_b'] + d['value_c']))
    ['check']
    .all()
)

print('✅ Données hiérarchiques générées:')
print(f'   Format wide : {df_hier.shape} (MultiIndex entity/date, {n_entities} entités)')
print(f'   Format long  : {df_hier.shape} (MultiIndex entity/category/component/date)')
print(f'   Vérification : value_a == value_b + value_c → {constraint_ok}')

print('\n📋 Aperçu format wide :')
print(df_hier.head(8))

### Création des prédictions

In [ ]:
# Étape 1 : Préparation des features pour chaque composante
# Construction de covariables simples (trend, mois, lags) pour chaque série

# Fonction de construction des features pour la prédiction de chaque valeur
def build_features(series: pd.Series, horizon: int) -> tuple[pd.DataFrame, pd.Series]:
    """Build lag-based features for a univariate time series.

    Args:
        series: Time series with a DatetimeIndex or a MultiIndex whose last
            level is a DatetimeIndex (panel format).
        horizon: Forecast horizon (number of steps ahead).

    Returns:
        Tuple of (X, y) aligned for the given horizon, with NaN rows dropped.
    """
    # Initialisation du DataFrame de features
    df_feat = pd.DataFrame(index=series.index)

    # Création des features temporelles et des lags
    df_feat["trend"] = np.arange(len(series))
    df_feat["month"] = series.index.get_level_values(-1).month
    for lag in [1, 2, 3, 6, 12]:
        df_feat[f"lag_{lag}"] = series.shift(lag)

    # Alignement de y[t+horizon] avec X[t]
    y_aligned = series.shift(-horizon)

    # Suppression des lignes incomplètes
    mask = df_feat.notna().all(axis=1) & y_aligned.notna()
    return df_feat.loc[mask], y_aligned.loc[mask]

# Fonction d'agrégation par entité à une fréquence donnée
def aggregate_panel(series: pd.Series, freq: str) -> pd.Series:
    """Aggregate a monthly panel series to a lower temporal frequency.

    Groups by entity and target period, summing monthly values within each
    period.  The resulting index uses period-start timestamps so that
    ``build_features`` and ``PanelOutOfSampleSplit`` remain compatible.

    Args:
        series: Monthly panel series with a MultiIndex (entity, date).
        freq: Pandas offset alias for the target frequency, e.g. ``"QS"``
            (quarter-start) or ``"YS"`` (year-start).

    Returns:
        Aggregated panel series with the same MultiIndex names (entity, date).
    """
    # Groupement par entité et période cible, puis sommation
    return (
        series
        .groupby("entity")
        .apply(func=lambda x : x.droplevel('entity').resample(freq).sum())
    )


# Horizon de prévision par fréquence
horizon_m = 2   # mensuel  : 2 mois
horizon_q = 1   # trimestriel : 1 trimestre
horizon_y = 1   # annuel      : 1 an

# Features mensuelles (hiérarchie cross-sectionnelle : a = b + c, c = d + e)
X_b, y_b = build_features(series=df_hier["value_b"], horizon=horizon_m)
X_d, y_d = build_features(series=df_hier["value_d"], horizon=horizon_m)
X_e, y_e = build_features(series=df_hier["value_e"], horizon=horizon_m)
X_c, y_c = build_features(series=df_hier["value_c"], horizon=horizon_m)
X_a, y_a = build_features(series=df_hier["value_a"], horizon=horizon_m)

# Agrégation temporelle du total vers fréquences trimestrielle et annuelle
value_a_q = aggregate_panel(df_hier["value_a"], freq="QS")
value_a_y = aggregate_panel(df_hier["value_a"], freq="YS")

X_a_q, y_a_q = build_features(series=value_a_q, horizon=horizon_q)
X_a_y, y_a_y = build_features(series=value_a_y, horizon=horizon_y)

# Application du décalage d'horizon sur les features (double décalage intentionnel)
X_a   = X_a.shift(-horizon_m)
X_b   = X_b.shift(-horizon_m)
X_c   = X_c.shift(-horizon_m)
X_d   = X_d.shift(-horizon_m)
X_e   = X_e.shift(-horizon_m)
X_a_q = X_a_q.shift(-horizon_q)
X_a_y = X_a_y.shift(-horizon_y)

# Affichage
print("Covariables mensuelles   :", X_b.shape, X_d.shape, X_e.shape, X_c.shape, X_a.shape)
print("Covariables trimestrielles:", X_a_q.shape)
print("Covariables annuelles     :", X_a_y.shape)

In [ ]:
# Étape 2 : Génération de prévisions out-of-sample via cross_val_predict
# Importation du modèle de prévision
from sklearn.linear_model import LinearRegression
from sklearn.utils import _safe_indexing

# Fonction d'itération d'une crossvall sur des listes de jeux de données
def run_cv_loop(
    cv_split,
    Xs_train: list[pd.DataFrame],
    ys_train: list[pd.Series],
    Xs_test: list[pd.DataFrame],
    full_index: pd.MultiIndex,
    col_names: list[str],
) -> pd.DataFrame:
    """Run a manual cross-validation loop and collect out-of-sample predictions.

    Args:
        cv_split: Cross-validator yielding (train_idx, test_idx) pairs.
        Xs_train: List of feature matrices to train on (one per target).
        ys_train: List of target series aligned with Xs_train.
        Xs_test: List of feature matrices to predict from (one per target).
        full_index: MultiIndex of the full dataset, used to reconstruct the
            (entity, date) index of each test fold.
        col_names: Column names for the returned DataFrame, one per target.

    Returns:
        DataFrame of predictions indexed by (entity, date).
    """
    model = LinearRegression()
    records = []

    for train_idx, test_idx in cv_split:
        # Prédiction indépendante pour chaque cible
        y_hats = []
        for X_tr_full, y_tr_full, X_te_full in zip(Xs_train, ys_train, Xs_test):
            X_tr = _safe_indexing(X_tr_full, train_idx)
            y_tr = _safe_indexing(y_tr_full, train_idx)
            X_te = _safe_indexing(X_te_full, test_idx)
            y_hats.append(model.fit(X_tr, y_tr).predict(X_te))

        # Reconstruction de l'index (entity, date) du fold de test
        test_index = full_index[test_idx]
        for (entity, date), *vals in zip(test_index, *y_hats):
            records.append({"entity": entity, "date": date, **dict(zip(col_names, vals))})

    return pd.DataFrame(records).set_index(["entity", "date"])


# Fenêtres de test par fréquence
test_dates_m = [d.strftime("%Y-%m-%d") for d in pd.date_range("2021-01-01", periods=24, freq="MS")]
test_dates_q = [d.strftime("%Y-%m-%d") for d in pd.date_range("2021-01-01", periods=8,  freq="QS")]
test_dates_y = [d.strftime("%Y-%m-%d") for d in pd.date_range("2021-01-01", periods=2,  freq="YS")]

# Cross-validateurs pour chaque fréquence
cv_m = PanelOutOfSampleSplit(test_indices=test_dates_m, test_size=1, gap=horizon_m)
cv_q = PanelOutOfSampleSplit(test_indices=test_dates_q, test_size=1, gap=horizon_q)
cv_y = PanelOutOfSampleSplit(test_indices=test_dates_y, test_size=1, gap=horizon_y)

# Prévisions mensuelles (hiérarchie cross-sectionnelle)
df_hat_m = run_cv_loop(
    cv_split=cv_m.split(X_a, y_a),
    Xs_train=[X_a, X_b, X_c, X_d, X_e],
    ys_train=[y_a, y_b, y_c, y_d, y_e],
    Xs_test=[X_a, X_b, X_c, X_d, X_e],
    full_index=X_a.index,
    col_names=["value_a", "value_b", "value_c", "value_d", "value_e"],
)

# Prévisions trimestrielles (total agrégé)
df_hat_q = run_cv_loop(
    cv_split=cv_q.split(X_a_q, y_a_q),
    Xs_train=[X_a_q],
    ys_train=[y_a_q],
    Xs_test=[X_a_q],
    full_index=X_a_q.index,
    col_names=["value_a_q"],
)

# Prévisions annuelles (total agrégé)
df_hat_y = run_cv_loop(
    cv_split=cv_y.split(X_a_y, y_a_y),
    Xs_train=[X_a_y],
    ys_train=[y_a_y],
    Xs_test=[X_a_y],
    full_index=X_a_y.index,
    col_names=["value_a_y"],
)

# Diagnostics
# Incohérence cross-sectionnelle (hiérarchie a = b + c, fréquence mensuelle)
incoherence_cs_a = (df_hat_m["value_a"] - (df_hat_m["value_b"] + df_hat_m["value_c"])).abs()
# Incohérence cross-sectionnelle (hiérarchie a = b + c, fréquence mensuelle)
incoherence_cs_c = (df_hat_m["value_c"] - (df_hat_m["value_d"] + df_hat_m["value_e"])).abs()

# Incohérences temporelles 
# Agrégation des prévisions mensuelles et trimestrielles vers les fréquences supérieures
y_hat_a_m2q = aggregate_panel(df_hat_m["value_a"], freq="QS")
y_hat_a_m2y = aggregate_panel(df_hat_m["value_a"], freq="YS")
y_hat_a_q2y = aggregate_panel(df_hat_q["value_a_q"], freq="YS")

# Calcul des écarts après alignement sur l'index commun
incoherence_m2q = (y_hat_a_m2q - df_hat_q["value_a_q"]).abs().dropna()
incoherence_m2y = (y_hat_a_m2y - df_hat_y["value_a_y"]).abs().dropna()
incoherence_q2y = (y_hat_a_q2y - df_hat_y["value_a_y"]).abs().dropna()

# Affichage
print("=== Prévisions mensuelles (cross-section) ===")
print(f"  Shape : {df_hat_m.shape}")
print(f"  Incohérence moyenne |ŷ_a - (ŷ_b + ŷ_c)| = {incoherence_cs_a.mean():.4f}")
print(f"  Incohérence max                          = {incoherence_cs_a.max():.4f}")
print(f"  Incohérence moyenne |ŷ_c - (ŷ_d + ŷ_e)| = {incoherence_cs_c.mean():.4f}")
print(f"  Incohérence max                          = {incoherence_cs_c.max():.4f}")

print("\n=== Prévisions trimestrielles (agrégation temporelle) ===")
print(f"  Shape : {df_hat_q.shape}")
print("  Mensuel → Trimestriel :")
print(f"    Incohérence moyenne = {incoherence_m2q.mean():.4f}")
print(f"    Incohérence max     = {incoherence_m2q.max():.4f}")

print("\n=== Prévisions annuelles (agrégation temporelle) ===")
print(f"  Shape : {df_hat_y.shape}")
print("  Mensuel → Annuel :")
print(f"    Incohérence moyenne = {incoherence_m2y.mean():.4f}")
print(f"    Incohérence max     = {incoherence_m2y.max():.4f}")
print("  Trimestriel → Annuel :")
print(f"    Incohérence moyenne = {incoherence_q2y.mean():.4f}")
print(f"    Incohérence max     = {incoherence_q2y.max():.4f}")

## Initialisation des arguments de la classe

In [ ]:
from hierarchicalforecast.methods import BottomUp, MinTrace

# Initialisation des arguments
# Méthodes de réconciliation
reconcilers=[BottomUp(), MinTrace(method='ols')]
# Hierarchie à respecter
hierarchy = {
    'value_a' : ['value_b', 'value_c'],
    'value_c' : ['value_d', 'value_e']
}
# Type d'agrégation ('local', 'global') pour l'agrégation temporelle et 'cross-section' sinon
aggregation_type = 'cross-section'
exog_vars = {'gdp' : 'mean' } # None
# Arguments de la méthode fix
X = df_hat_m.copy()
y = df_hier.loc[df_hat_m.index].copy()
# Ajout de variables exogènes à X pour tester
X['gdp'] = np.random.normal(0, 1.0, len(X))

In [ ]:
X.head()

In [ ]:
y.head()

## Fonctions utilitaires de traitement de la hiérarchie

In [ ]:
# Méthode auxiliaire de recherche des feuilles
def _find_leaves(hierarchy: dict[str, list[str]]) -> list[str]:
    """Return leaf nodes (those with no children in the hierarchy).
 
    Args:
        hierarchy: Mapping from parent component to its direct children.
 
    Returns:
        Sorted list of leaf node names.
    """
    # Extraction des noeuds internes
    internal_nodes = set(hierarchy.keys())
    # Extraction des feuilles
    leaves = set(np.concatenate(list(hierarchy.values())).tolist()) - set(hierarchy.keys())

    return sorted(leaves)
 
# Méthode auxiliaire de construction des branches parent-enfant
def _build_parent_map(hierarchy: dict[str, list[str]]) -> dict[str, str]:
    """Build a child → parent mapping from the hierarchy dict.
 
    Args:
        hierarchy: Mapping from parent component to its direct children.
 
    Returns:
        Dict mapping each non-root node to its direct parent.
    """
    return {
        child: parent
        for parent, children in hierarchy.items()
        for child in children
    }
 
# Méthode auxiliaire de description du chemin entre un noeud et la racine
def _path_from_root(node: str, parent_map: dict[str, str]) -> list[str]:
    """Return the path from root to node, excluding the root, top-down.
 
    Args:
        node: Target node name.
        parent_map: Dict mapping each node to its direct parent.
 
    Returns:
        Ordered list of node names from the first child of root down to
        ``node`` (inclusive).  Empty list if ``node`` is the root itself.
    """
    # Remontée vers la racine puis inversion
    path: list[str] = []
    current = node
    while current in parent_map:
        path.append(current)
        current = parent_map[current]
    # Ajout de la racine (dernier nœud sans parent)
    path.append(current)
    return list(reversed(path))


## Débug de la méthode "fit"

In [ ]:
# Reformattage des données d'entrée et création des specs
# Traitement de la hiérarchie
# Identification des feuilles
leaves = _find_leaves(hierarchy=hierarchy)
# Identification des branches
parent_map = _build_parent_map(hierarchy=hierarchy)
# Identification de l'ensemble des noeuds
all_nodes = set(np.concatenate(list(hierarchy.values())).tolist()) | set(hierarchy.keys())

# Chemins feuille → liste de noeuds depuis la racine (racine exclue)
leaf_paths: dict[str, list[str]] = {
    leaf: _path_from_root(leaf, parent_map) for leaf in leaves
}
# Calcul de la profondeur maximale
max_depth = max(len(p) for p in leaf_paths.values())

# Noms de colonnes pour chaque niveau
level_cols = [f"_hierarchical_level{i}" for i in range(max_depth)]

# Table de correspondance feuille → chemin padé (répétition du dernier élément pour avoir des éléments de profondeur homogène)
leaf_to_padded: dict[str, list[str]] = {
    leaf: path + [path[-1]] * (max_depth - len(path))
    for leaf, path in leaf_paths.items()
}

# Construction du DataFrame associant à chaque colonne son nom dans la hiérarchie
path_lookup = (
    pd.DataFrame.from_dict(leaf_to_padded, orient="index", columns=level_cols)
    .rename_axis("leaf")
    .reset_index()
)
path_lookup.head()

In [ ]:
leaf_to_padded

In [ ]:
path_lookup

## Débug de la méthode "predict"